In [ ]:
import fitz
from langchain_core.documents import  Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.prompts import PromptTemplate
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

A CLIP model (Contrastive Language–Image Pretraining) is a multimodal AI model developed by OpenAI that understands both images and text in the same space.

It can connect images and text meaningfully — like “this picture matches this sentence.

In [67]:
from dotenv import load_dotenv
load_dotenv()

# setup the environment
#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# initialize the clip model for unified embeddingsa
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32") # both text and image embeddings
# this processor will helps to convert both text and images into the format required by the CLIP model
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()  # set the model to evaluation mode

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 10771.42it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05, eleme

Example (real-world ML)
Image model → Normalization (0–1)
Regression model → Standardization

CLIP embeddings → L2 normalization

🔹 Quick decision rule (easy to remember)
👉 Deep learning / images → Normalization

👉 Distance-based models → Normalization

👉 Statistical models → Standardization

Problem without normalization

Suppose you have two embeddings:

Text A → [2, 2]  
Text B → [10, 10]

👉 Both mean the same thing (same direction)
BUT:

Text B has larger values
So it looks “more important”

👉 ❌ Wrong comparison

🔹 After normalization
Text A → [0.7, 0.7]  
Text B → [0.7, 0.7]

👉 Now:

Same direction
Same length
Correctly treated as similar
🔹 Why this is critical in CLIP

CLIP compares:

text embeddings
image embeddings

👉 Using cosine similarity

🔥 Key point

Cosine similarity cares about:

direction, not magnitude

👉 Normalization removes magnitude

In [46]:
def embed_image(image_path):
    """Embed image data using CLIP"""
    if isinstance(image_path , str): # if its a image path
        image = Image.open(image_path).convert("RGB")
    else:
        image = image_path # if its am image
        
    # we convert the image to the format required by CLIP and get the pytorch tensor representation
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
                                        # inputs will contain 2 or 3 dimensions ** means open dimensions and pass the values
        features = model.get_image_features(**inputs)

        # every image dim is different to convert into a unit vector 
        features= features / features.norm(dim = -1, keepdim=True)  # normalize the features
        return features.squeeze().numpy()  # convert to numpy array for easier handling

def embed_text(text):
    """Embed text using CLIP"""
    inputs = processor(text= text , return_tensors="pt", padding=True, truncation=True , max_length=77) # CLIP has a max token limit of 77 for text
    with torch.no_grad():
        features = model.get_text_features(**inputs)
        #We normalize embeddings so that no vector is preferred just because it has a larger magnitude
        features = features / features.norm(dim=-1, keepdim=True)  # normalize the features
        return features.squeeze().numpy()  # convert to numpy array for easier handling

🔹 1. What is processor?
👉 Definition

processor is a preprocessing tool that prepares your input (text or image) in the format the CLIP model expects.

🔹 2. What is model.get_text_features()?
👉 Definition

It is a function that:

Takes processed text → passes it through CLIP → returns a feature vector (embedding)

In [47]:
# Process PDF
pdf_path = "C:\\Users\\ASUS TUFF\\Desktop\\AgenticAI\\test_multimodal_rag.pdf"
doc = fitz.open(pdf_path)

# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} # store actual image data for LLM

# text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50) # split text into chunks of 500 characters with 50 characters overlap
 

In [48]:
doc

Document('C:\Users\ASUS TUFF\Desktop\AgenticAI\test_multimodal_rag.pdf')

🔹 What this creates (example)

Suppose page 0 has:

"This is a sample PDF"

👉 Then temp_doc becomes:

{
  "page_content": "This is a sample PDF",
  "metadata": {
      "page": 0,
      "type": "text"
  }
}
🔹 Why not just use text?

Good question.

👉 You could use plain text, but:

❌ Problem:
You lose:
page number
type (text/image)
source info
✅ With Document:

You keep everything together

In [49]:
import torch.nn.functional as F

def embed_image(image_path):
    if isinstance(image_path, str):
        image = Image.open(image_path).convert("RGB")
    else:
        image = image_path

    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"]

    with torch.no_grad():
        output = model.get_image_features(pixel_values=pixel_values)
        
        # ✅ Extract the actual tensor from the object
        if hasattr(output, "pooler_output"):
            features = output.pooler_output
        elif hasattr(output, "last_hidden_state"):
            features = output.last_hidden_state[:, 0, :]  # CLS token
        else:
            features = output  # already a tensor
        
        features = F.normalize(features, p=2, dim=-1)
        return features.squeeze().numpy()


def embed_text(text):
    inputs = processor(text=text, return_tensors="pt", padding=True, truncation=True, max_length=77)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        output = model.get_text_features(input_ids=input_ids, attention_mask=attention_mask)
        
        # ✅ Extract the actual tensor from the object
        if hasattr(output, "pooler_output"):
            features = output.pooler_output
        elif hasattr(output, "last_hidden_state"):
            features = output.last_hidden_state[:, 0, :]
        else:
            features = output  # already a tensor
        
        features = F.normalize(features, p=2, dim=-1)
        return features.squeeze().numpy()

Simple answer:
This code converts an image into a text string so you can store and send it easily.
Why?
Later in your RAG pipeline, when you find a relevant image, you need to send it to GPT-4V for analysis. GPT-4V doesn't accept image files directly — it only accepts base64 encoded strings.
What each line does:

pythonbuffered = io.BytesIO()                          # creates a temporary memory buffer (like a fake file)
pil_image.save(buffered, format="PNG")           # saves the PIL image into that buffer
img_base64 = base64.b64encode(buffered.getvalue()).decode()  # converts bytes → base64 string
image_data_store[image_id] = img_base64          # stores it with a unique id for later retrieval
Real world analogy:
Imagine you have a physical photo. You can't send it through a text message directly — so you scan it and convert it to text characters. That's exactly what base64 does to an image.
The flow in your project:
PDF image → PIL Image → base64 string → stored in image_data_store
                                                    ↓
                              later retrieved and sent to GPT-4V
So basically you're pre-processing and caching all images upfront so when retrieval happens, you can instantly send them to GPT-4V without reprocessing.

In [50]:
for i , page in enumerate(doc):
    # Process text
    text = page.get_text()
    if text.strip(): # checks if text exists
        # Create a temporary document for splitting
        temp_doc = Document(page_content= text , metadata={"page": i , "type": "text"})
        # this temp_doc will save only 1 page text 
        text_chunks = splitter.split_documents([temp_doc]) # split the text into chunks splitter_document expects list

        # embed each chunk and store
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)  

    # Process images
    # Thress Important Actions

    # Convert PDF Image to PIL Image
    # Store as base64 for GPT-4v (Which needs base64 encoded images)
    # Create CLIP embeddings for retrieval
                                        
                                    # .get_images will return an image id , height , weidth and some info
    for img_index , img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"

            # store image as base64 for later use with GPT-4v
            buffered = io.BytesIO()
            pil_image.save(buffered , format = "PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64

            # Embed image using CLIP
            embedding = embed_image(pil_image)

            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata = {"page":i , "type":"image" , "image_id": image_id}
            )
            all_docs.append(image_doc)
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()
    


🔹 What each value means

For this tuple:

(12, 0, 300, 200, 8, 'DeviceRGB', '', 'Im1', 'DCTDecode')
Position	Meaning
12	✅ Image ID (xref)
300	✅ Width
200	✅ Height
others	extra internal info (not usually needed)

In [51]:
all_embeddings

[array([ 2.97035426e-02,  3.16164009e-02, -2.04469310e-03, -3.53704616e-02,
        -6.41318178e-03,  1.99033152e-02, -1.30402539e-02,  9.36409682e-02,
         1.05504230e-01,  3.61796259e-03,  4.49166261e-02,  4.89348546e-02,
         1.28451856e-02,  9.62477876e-04,  1.72885992e-02,  2.47358494e-02,
        -1.26107465e-02,  1.77272670e-02, -4.62155938e-02, -1.41253155e-02,
        -2.05464140e-02,  1.18219955e-02, -7.03781005e-03,  8.69671069e-03,
        -2.27239113e-02,  2.01372784e-02, -8.23047757e-03,  2.40751132e-02,
        -2.83966251e-02, -3.83285992e-02, -6.08844333e-04,  7.25441379e-04,
         1.77693088e-02, -9.05888155e-03, -1.13100661e-02, -4.46463227e-02,
        -2.30448861e-02,  2.11335849e-02,  4.91218455e-02, -4.65038605e-02,
        -2.37824544e-02, -2.68438627e-04, -2.43784934e-02,  3.61414161e-03,
         2.50972323e-02, -1.59085020e-02, -1.74102597e-02, -3.72038558e-02,
        -4.29668166e-02,  3.57386209e-02, -6.95410743e-03, -7.43085038e-05,
         2.1

In [52]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Multimodal RAG Test Document\nThis is a sample PDF document designed to test multimodal retrieval-augmented generation (RAG)\nsystems. It contains: - Text data - A simple visual element You can test: 1. Text retrieval 2. Embedding\nextraction 3. Cross-modal understanding\nSample Image Block')]

In [53]:
# Create unified FAISS Vector Store with CLIP Embeddings

embeddings_array = np.array(all_embeddings)
embeddings_array

array([[ 2.97035426e-02,  3.16164009e-02, -2.04469310e-03,
        -3.53704616e-02, -6.41318178e-03,  1.99033152e-02,
        -1.30402539e-02,  9.36409682e-02,  1.05504230e-01,
         3.61796259e-03,  4.49166261e-02,  4.89348546e-02,
         1.28451856e-02,  9.62477876e-04,  1.72885992e-02,
         2.47358494e-02, -1.26107465e-02,  1.77272670e-02,
        -4.62155938e-02, -1.41253155e-02, -2.05464140e-02,
         1.18219955e-02, -7.03781005e-03,  8.69671069e-03,
        -2.27239113e-02,  2.01372784e-02, -8.23047757e-03,
         2.40751132e-02, -2.83966251e-02, -3.83285992e-02,
        -6.08844333e-04,  7.25441379e-04,  1.77693088e-02,
        -9.05888155e-03, -1.13100661e-02, -4.46463227e-02,
        -2.30448861e-02,  2.11335849e-02,  4.91218455e-02,
        -4.65038605e-02, -2.37824544e-02, -2.68438627e-04,
        -2.43784934e-02,  3.61414161e-03,  2.50972323e-02,
        -1.59085020e-02, -1.74102597e-02, -3.72038558e-02,
        -4.29668166e-02,  3.57386209e-02, -6.95410743e-0

In [54]:
# Create unified FAISS Vector Store with CLIP Embeddings

# Create custom FAISS index since we have precomputed embeddings
Vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content , emb) for doc , emb in zip(all_docs , embeddings_array)] ,
    embedding= None, # we are using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
    )


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [56]:
def retrive_multimodel(query , k = 5):
    """Unified retrieval using CLIP embeddings for both text and images"""
    # Embed query using CLIP
    query_embedding = embed_text(query)

    # Search in unified vector store
    results = Vector_store.similarity_search_by_vector(
        embedding=query_embedding ,
        k = k
    )
    return results

In [57]:
def create_multimodal_message(query, retrieved_docs):
    content = []

    # add query
    content.append({"type": "text", "text": f"Question: {query}\n\n"})

    # loop through all retrieved docs
    for doc in retrieved_docs:
        if doc.metadata["type"] == "text":
            # add text directly
            content.append({"type": "text", "text": doc.page_content})

        elif doc.metadata["type"] == "image":
            # get base64 and add image
            image_id = doc.metadata["image_id"]
            img_base64 = image_data_store[image_id]
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_base64}"}
            })

    return content

In [70]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def rag_pipeline(query):
    retrieved_docs = retrive_multimodel(query, k=5)
    content = create_multimodal_message(query, retrieved_docs)

    response = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",  # groq vision model
        messages=[{"role": "user", "content": content}]
    )
    return response.choices[0].message.content

In [ ]:
query = "What is that Image"
a = retrive_multimodel(query=query , k=1)
b = rag_pipeline(query)


In [72]:
b

"The image you're referring to is likely a visual element embedded within a sample PDF document. This document is designed to test the capabilities of multimodal retrieval-augmented generation (RAG) systems. \n\nGiven the description, the image is probably a simple visual element, possibly a:\n\n1. **Diagram**: A basic illustration used to support the text or to provide a visual representation of a concept.\n2. **Chart or Graph**: A simple graphical representation of data.\n3. **Picture or Icon**: A basic image used to add visual interest or to illustrate a point.\n\nWithout the actual image, it's difficult to provide a more specific identification. However, the purpose of including such an image in the document is to test the system's ability to:\n\n- **Retrieve text**: Understand and extract information from the written content.\n- **Extract embeddings**: Derive meaningful representations (embeddings) from the text and possibly the image.\n- **Cross-modal understanding**: Combine inf